In [ ]:
# === Dependencies ===
import os
import re
import json
import time
import unicodedata
import fitz  # PyMuPDF
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from PyPDF2 import PdfReader

# === OpenAI Client ===
os.environ["OPENAI_API_KEY"] = ""
client = OpenAI()

# === Settings ===
GPT_MODEL = "gpt-4o"

PDF_FOLDER = "paper"
OUTPUT_EXCEL = "cr3_spectra_database.xlsx"

REQUIRED_KEYS = [
    "Formula",
    "Cr_conc",
    "Emission_nm",
    "FWHM_nm",
    "DOI"
]

# =====================================================================
# Prompt
# =====================================================================

SYSTEM_PROMPT = """
You are an expert assistant extracting spectroscopic data from Cr3+-activated phosphor research papers.

Your task is to extract one JSON object for every unique Cr3+ phosphor composition reported in the paper.

Extract ONLY the following information.

1. Formula
- Full phosphor composition including Cr3+.
- Example:
  Ba2ScTaO6:Cr3+

2. Cr_conc
- Cr3+ concentration corresponding to the reported spectra.
- Example:
  1 mol%
  0.5 mol%
  x = 0.02

3. Emission_nm
- Room-temperature emission peak position(s) in nm.
- If multiple peaks exist, separate using commas.
- Ignore excitation wavelengths.

4. FWHM_nm
- Full width at half maximum (FWHM) of the corresponding emission peak(s).
- If multiple values exist, keep the same order as Emission_nm.

Rules:

• Extract ONLY explicitly reported values.
• Never estimate missing values.
• Return null if unavailable.
• Create one JSON object for every unique composition/concentration.
• Return ONLY valid JSON.

Example:

[
{
"Formula":"Ba2ScTaO6:Cr3+",
"Cr_conc":"1 mol%",
"Emission_nm":"890",
"FWHM_nm":"194"
},
{
"Formula":"LiInW2O8:Cr3+",
"Cr_conc":"1 mol%",
"Emission_nm":"1085",
"FWHM_nm":"265"
}
]
"""

USER_PROMPT_TEMPLATE = """
Extract Cr3+ phosphor spectroscopic data from the following paper.

{text}

Return only the JSON array.
"""

# =====================================================================
# Helpers
# =====================================================================

def clean_text(text):
    return unicodedata.normalize(
        "NFKD",
        text
    ).encode(
        "utf-8",
        "ignore"
    ).decode(
        "utf-8",
        "ignore"
    )


def extract_text_excluding_back_matter(pdf_path):

    doc = fitz.open(pdf_path)

    full_text = "".join(
        page.get_text()
        for page in doc
    )

    text_lower = full_text.lower()

    stop_keywords = [
        "references",
        "acknowledgements",
        "funding",
        "conflicts of interest",
        "author contributions"
    ]

    stop_pos = len(full_text)

    for keyword in stop_keywords:

        pos = text_lower.find(keyword)

        if 0 <= pos < stop_pos:
            stop_pos = pos

    return clean_text(
        full_text[:stop_pos].strip()
    )


def extract_doi(pdf_path):

    try:

        reader = PdfReader(pdf_path)

        first_page = reader.pages[0].extract_text()

        match = re.search(
            r"10\.\d{4,9}/[-._;()/:A-Z0-9]+",
            first_page,
            re.I
        )

        if match:
            return match.group(0)

    except:
        pass

    return None


def clean_reply(reply):

    return (
        clean_text(reply)
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )


# =====================================================================
# GPT Call
# =====================================================================

def call_gpt(text, filename):

    prompt = USER_PROMPT_TEMPLATE.format(text=text)

    try:

        response = client.chat.completions.create(

            model=GPT_MODEL,

            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],

            temperature=0.2

        )

        reply = clean_reply(
            response.choices[0].message.content
        )

        return json.loads(reply)

    except Exception as e:

        print(f"❌ {filename}: {e}")

        return None


# =====================================================================
# Main
# =====================================================================

def main():

    files = sorted(
        f for f in os.listdir(PDF_FOLDER)
        if f.endswith(".pdf")
    )

    all_rows = []

    total_time = 0

    print(f"Found {len(files)} PDFs\n")

    for i, fname in enumerate(files, 1):

        print(f"\n[{i}/{len(files)}] {fname}")

        path = os.path.join(
            PDF_FOLDER,
            fname
        )

        text = extract_text_excluding_back_matter(path)

        if not text.strip():

            print("Empty PDF")

            continue

        doi = extract_doi(path)

        start = time.time()

        result = call_gpt(text, fname)

        elapsed = time.time() - start

        total_time += elapsed

        if result:

            for entry in result:

                entry["DOI"] = doi
                entry["Source_File"] = fname

                all_rows.append(entry)

                print(
                    json.dumps(
                        entry,
                        indent=2
                    )
                )

            print(
                f"✓ {len(result)} entries ({elapsed:.1f}s)"
            )

        else:

            print("Extraction failed")

    df = pd.DataFrame(all_rows)

    for col in REQUIRED_KEYS + ["Source_File"]:

        if col not in df.columns:
            df[col] = None

    df = df[
        REQUIRED_KEYS + ["Source_File"]
    ]

    df.to_excel(
        OUTPUT_EXCEL,
        index=False
    )

    print("\nDone.")
    print(df.head())
    print(f"\nSaved to {OUTPUT_EXCEL}")


# =====================================================================

if __name__ == "__main__":

    main()